# ETH Demo Notebook: 5 Spin-Chain Models

This notebook demonstrates ETH diagonal elements for:

1. Transverse-field Ising chain (TFIM)
2. XXZ chain
3. Disordered Heisenberg chain
4. Random Pauli-string Hamiltonian
5. A custom model built directly with `eth_spin_chain.py` API

For each model, we plot diagonal ETH matrix elements $\langle E_i|O|E_i\rangle$ versus energy $E_i$.

## Setup

In Google Colab, connect to a **CPU** runtime and choose **Runtime > Run all**.
The next cell installs this package and its dependencies from the public
repository's `main` branch. No GPU or Google Drive connection is needed.

For local Jupyter use, first run `python3 -m pip install -e .` from the repository
root and select a kernel using that Python environment. The installation cell
below runs only in Colab.

Each model cell defines its own chain length. Most examples use `L=6` or
`L=8`; the disordered Heisenberg example uses `L_dis=11` to reproduce the
representative result in the README. Set `L_dis=8` for a faster first run.
After running the notebook once, try changing a model's chain length, coupling
constants, random seed, or observable, then rerun that cell. Increasing `L`
raises the computation and memory cost exponentially.

In [ ]:
try:
    import google.colab
except ImportError:
    pass  # For local use, install the project before opening this notebook.
else:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "git+https://github.com/osamufukushima/ETH-spin-chain.git@main",
    ])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from eth_spin_chain import (
    DiagonalizedHamiltonian,
    ObservableSpec,
    ObservableTerm,
    InteractionTerm,
    HamiltonianSpec,
    sample_interaction_terms,
    sampled_interaction_hamiltonian,
    all_site_product_observable,
    eth_elements,
)
from standard_models import (
    transverse_field_ising_chain,
    xxz_chain,
    disordered_heisenberg_chain,
    random_pauli_hamiltonian,
)

plt.rcParams['figure.figsize'] = (6.5, 4.2)
plt.rcParams['axes.grid'] = True

def plot_eth_diagonal_for_model(
    title, ham, obs, color='tab:blue',
    observable_label=r'$\langle E_i|O|E_i\rangle$',
):
    prepared = DiagonalizedHamiltonian.from_spec(ham)
    result = eth_elements(prepared, obs)

    fig, ax = plt.subplots()
    ax.scatter(result.energies, np.real(result.diagonal), s=5, alpha=0.5, c=color, edgecolors='none')
    ax.set_title(f"{title} (L={ham.L})")
    ax.set_xlabel('Energy $E_i$')
    ax.set_ylabel(observable_label)
    fig.tight_layout()
    plt.show()

    return prepared, result

## 1) Transverse-Field Ising Chain (TFIM)

Hamiltonian:
$$
H_{\mathrm{TFIM}} = -j \sum_i Z_i Z_{i+1} - h \sum_i X_i - g \sum_i Z_i
$$

Observable:
$$
O = Z_0 Z_1
$$

In [ ]:
L_tfim = 8
ham_tfim = transverse_field_ising_chain(L=L_tfim, j=1.0, h=1.05, g=0.2, boundary='periodic')
obs_tfim = ObservableSpec([ObservableTerm(sites=[0, 1], ops='ZZ')])
_tfim_prepared, _tfim_res = plot_eth_diagonal_for_model(
    'TFIM: ETH diagonal elements',
    ham_tfim,
    obs_tfim,
    color='tab:green',
    observable_label=r'$\langle E_i|Z_0 Z_1|E_i\rangle$',
)

## 2) XXZ Chain

Hamiltonian:
$$
H_{\mathrm{XXZ}} = j_{xy} \sum_i (X_iX_{i+1} + Y_iY_{i+1}) + j_z \sum_i Z_iZ_{i+1}
$$

Observable:
$$
O = Z_0 Z_1
$$

In [ ]:
L_xxz = 8
ham_xxz = xxz_chain(L=L_xxz, jxy=1.0, jz=0.7, boundary='periodic')
obs_xxz = ObservableSpec([ObservableTerm(sites=[0, 1], ops='ZZ')])
_xxz_prepared, _xxz_res = plot_eth_diagonal_for_model(
    'XXZ: ETH diagonal elements',
    ham_xxz,
    obs_xxz,
    color='tab:blue',
    observable_label=r'$\langle E_i|Z_0 Z_1|E_i\rangle$',
)

## 3) Disordered Heisenberg Chain

Hamiltonian:
$$
H_{\mathrm{dis.Heis}} = \sum_i J_i^{x} X_iX_{i+1} + \sum_i J_i^{y} Y_iY_{i+1} + \sum_i J_i^{z} Z_iZ_{i+1} + \sum_i (h_i^x X_i + h_i^y Y_i + h_i^z Z_i),
$$
with weak disorder ranges for each channel.

Observable:
$$
O = Z_0 Z_1
$$

This example uses `L_dis=11` (2048 eigenstates), `seed=123`, and the
nearest-neighbor observable $Z_0 Z_1$, matching the README representative
result. Its chain length is independent of the earlier model cells.
For a faster run, change `L_dis` to `8`.

In [ ]:
L_dis = 11
ham_dis = disordered_heisenberg_chain(
    L=L_dis,
    jx=(0.9, 1.1),
    jy=(0.9, 1.1),
    jz=(0.9, 1.1),
    hx=(-0.05, 0.05),
    hy=(-0.05, 0.05),
    hz=(-0.05, 0.05),
    seed=123,
)
obs_dis = ObservableSpec([ObservableTerm(sites=[0, 1], ops='ZZ')])
_dis_prepared, _dis_res = plot_eth_diagonal_for_model(
    'Disordered Heisenberg: ETH diagonal elements',
    ham_dis,
    obs_dis,
    color='red',
    observable_label=r'$\langle E_i|Z_0 Z_1|E_i\rangle$',
)

## 4) Random Pauli-String Hamiltonian

Hamiltonian:
$$
H_{\mathrm{rand}} = \sum_{k=1}^{n_{\mathrm{terms}}} c_k P_k,
$$
where $P_k$ are sampled Pauli strings and $c_k$ are sampled coefficients.

Observable:
$$
O = X_0 X_1
$$

In [ ]:
L_rand = 6
ham_rand = random_pauli_hamiltonian(
    L=L_rand,
    term_width=3,
    n_terms=12,
    coeff_min=-1.0,
    coeff_max=1.0,
    seed=7,
)
obs_rand = ObservableSpec([ObservableTerm(sites=[0, 1], ops='XX')])
_rand_prepared, _rand_res = plot_eth_diagonal_for_model(
    'Random Pauli Hamiltonian: ETH diagonal elements',
    ham_rand,
    obs_rand,
    color='tab:purple',
    observable_label=r'$\langle E_i|X_0 X_1|E_i\rangle$',
)

## 5) Custom Model Built Directly with `eth_spin_chain.py` API

This section uses API building blocks not used in the four standard demos:

- `sampled_interaction_hamiltonian(...)` to sample translated `XY` couplings
- direct manual term injection via `InteractionTerm`
- `all_site_product_observable(...)` for a nonlocal observable

Model:
$$
H = \sum_i J_i X_iY_{i+1} + g\sum_i Z_iZ_{i+1} + \lambda \sum_i X_i + \kappa X_0Y_1Z_2
$$
with sampled $J_i \in [0.8, 1.2]$.

Observable (nonlocal):
$$
O = \prod_{i=0}^{L-1} Z_i
$$

In [ ]:
L_custom = 8
base_xy = sampled_interaction_hamiltonian(
    L=L_custom,
    ops='XY',
    j_min=0.8,
    j_max=1.2,
    boundary='periodic',
    seed=2026,
)

zz_terms = sample_interaction_terms(
    L=L_custom,
    ops='ZZ',
    j_min=0.6,
    j_max=0.6,
    boundary='periodic',
)

x_field_terms = [InteractionTerm(sites=[i], ops='X', coeff=0.25) for i in range(L_custom)]
three_body = InteractionTerm(sites=[0, 1, 2], ops='XYZ', coeff=0.15)

ham_custom = HamiltonianSpec(
    L=L_custom,
    boundary='periodic',
    terms=[*base_xy.terms, *zz_terms, *x_field_terms, three_body],
)

obs_custom = all_site_product_observable(L=L_custom, ops='Z')
_custom_prepared, _custom_res = plot_eth_diagonal_for_model(
    'Custom API-built model: ETH diagonal elements',
    ham_custom,
    obs_custom,
    color='tab:orange',
    observable_label=r'$\langle E_i|\prod_j Z_j|E_i\rangle$',
)

## Notes

- Exact diagonalization scales exponentially with system size ($2^L$).
- For interactive use, start with small $L$ values (for example, $L=6$ to $L=10$).
- Each model defines its own chain length; changing one does not change the others.
- These plots use full-spectrum diagonal elements. No energy window or off-diagonal statistics are computed.
- To compare models, account for differences in chain length, observables, and symmetries.